## PST calculation

In [4]:
%pip install geopandas networkx matplotlib

  Using cached geopandas-1.1.3-py3-none-any.whl.metadata (2.3 kB)
  Using cached pyogrio-0.12.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (5.9 kB)
  Using cached pyproj-3.7.2-cp313-cp313-macosx_14_0_arm64.whl.metadata (31 kB)
  Using cached shapely-2.1.2-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.8 kB)
Using cached geopandas-1.1.3-py3-none-any.whl (342 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 42.8 MB/s eta 0:00:00a 0:00:01
Using cached pyogrio-0.12.1-cp313-cp313-macosx_12_0_arm64.whl (23.7 MB)
Using cached pyproj-3.7.2-cp313-cp313-macosx_14_0_arm64.whl (4.6 MB)
Using cached shapely-2.1.2-cp313-cp313-macosx_11_0_arm64.whl (1.6 MB)

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point

print("🌐 RUNNING MAC-NATIVE PYTHON GRAPH ENGINE (TRUE PST ALGORITHM SIMULATOR)...")

# ==============================================================================
# 1. ESTABLISH PATHWAYS FROM REPO TREE
# ==============================================================================
gpkg_path = "../data/processed/rotterdam_wijkenbuurten_enriched.gpkg"
highways_path = "../QGIS project files/Rotterdam Network/highways.shp"
boundary_path = "../QGIS project files/Rotterdam Shapefile/rotterdam.shp"

# Standardizing projections to Amersfoort / RD New (EPSG:28992) for metric calculations
print("-> Synthesizing and aligning spatial projections to EPSG:28992...")
boundary = gpd.read_file(boundary_path).to_crs(epsg=28992)
highways = gpd.read_file(highways_path).to_crs(epsg=28992)
buurten = gpd.read_file(gpkg_path, layer="buurten_enriched").to_crs(epsg=28992)

# ==============================================================================
# 2. GENERATE BGI ATTRACTION POINTS FROM HIGH-COOLING ZONES
# ==============================================================================
print("-> Synthesizing Blue-Green Infrastructure (BGI) core attraction centroids...")
# Isolate high-vegetation neighborhoods to find core cooling network hubs
high_green_zones = buurten[buurten['mean_NDVI'] > buurten['mean_NDVI'].median()]
bgi_attractions = high_green_zones.copy()
bgi_attractions['geometry'] = bgi_attractions.geometry.centroid

# ==============================================================================
# 3. CONSTRUCT TOPOLOGICAL NETWORK GRAPH MATRIX
# ==============================================================================
print("-> Building mathematical space syntax line-segment network graph...")
G = nx.Graph()

for idx, row in highways.iterrows():
    # Only map line strings with valid coordinate intersections
    if row.geometry and row.geometry.geom_type == 'LineString':
        coords = list(row.geometry.coords)
        start_node = coords[0]  # Intersection A
        end_node = coords[-1]   # Intersection B
        G.add_edge(start_node, end_node, id=idx, length=row.geometry.length)

print(f"   ✔ Graph complete: {G.number_of_nodes()} intersections, {G.number_of_edges()} street segments connected.")

# Snap attraction centroids directly to their closest network node coordinate
attraction_nodes = set()
for _, att in bgi_attractions.iterrows():
    closest_node = min(G.nodes, key=lambda n: att.geometry.distance(Point(n)))
    attraction_nodes.add(closest_node)

# ==============================================================================
# 4. EXECUTE PST ATTRACTION ACCESSIBILITY (TOPOLOGICAL DEPTH STEP SCORING)
# ==============================================================================
print("-> Simulating human navigation metrics (Counting step-turns to cool zones)...")
street_accessibility_scores = {}

# Define a maximum pedestrian search threshold (e.g., 3 topological intersections/turns away)
MAX_TURNS = 3 

for node in G.nodes:
    # Compute topological distance steps from current node to all reachable nodes
    lengths = nx.single_source_shortest_path_length(G, node, cutoff=MAX_TURNS)
    
    node_score = 0
    for target_node, steps in lengths.items():
        if target_node in attraction_nodes:
            # Replicate PST's mathematical gravity penalization: closer paths = higher score
            node_score += 1 / (steps + 1)
            
    # Attribute the highest intersection network value back to its adjacent street lines
    for edge in G.edges(node, data=True):
        street_id = edge[2]['id']
        if street_id not in street_accessibility_scores or node_score > street_accessibility_scores[street_id]:
            street_accessibility_scores[street_id] = node_score

highways['pst_accessibility_score'] = highways.index.map(street_accessibility_scores).fillna(0)

# ==============================================================================
# 5. AGGREGATE UP TO ADMINISTRATIVE NEIGHBORHOODS FOR QUARTO MODELING
# ==============================================================================
print("-> Grouping line segment values up into administrative CBS buurten...")
joined = gpd.sjoin(highways, buurten, how='left', predicate='intersects')
neighborhood_scores = joined.groupby('buurtnaam')['pst_accessibility_score'].mean().reset_index()

buurten_final = buurten.merge(neighborhood_scores, on='buurtnaam', how='left')
buurten_final['pst_accessibility_score'] = buurten_final['pst_accessibility_score'].fillna(0)

# ==============================================================================
# 6. EXPORT DIRECTLY INTO RAW GPKG METADATA ARCHITECTURE
# ==============================================================================
print("-> Exporting matrix layer back into master data pipeline architecture...")
buurten_final.to_file(gpkg_path, layer="buurten_clustered", driver="GPKG")

print("\n========================================================================")
print("✅ SUCCESS: PST Attraction Accessibility Engine completed.")
print("   Column 'pst_accessibility_score' is live in 'buurten_clustered'.")
print("========================================================================")

🌐 RUNNING MAC-NATIVE PYTHON GRAPH ENGINE (TRUE PST ALGORITHM SIMULATOR)...
-> Synthesizing and aligning spatial projections to EPSG:28992...
-> Synthesizing Blue-Green Infrastructure (BGI) core attraction centroids...
-> Building mathematical space syntax line-segment network graph...
   ✔ Graph complete: 2271 intersections, 2008 street segments connected.
-> Simulating human navigation metrics (Counting step-turns to cool zones)...
-> Grouping line segment values up into administrative CBS buurten...
-> Exporting matrix layer back into master data pipeline architecture...

✅ SUCCESS: PST Attraction Accessibility Engine completed.
   Column 'pst_accessibility_score' is live in 'buurten_clustered'.
